# U5-L4：语音会话管理

**课程目标**

1. 能按输入和输出解释采音、语音转写、回复生成、语音合成与播放，说明 `chat()` 如何支持持续问答。
2. 能说明任务句柄的用途，读取任务状态，判断任务是否结束。
3. 能画出会话管理的状态与转换条件，运行并调整程序，完成启动、阻止重复启动、停止和再次启动。


## 一、同一个会话中的连续问答

L1 观察了录音与语音识别，L2 将文字合成为语音，L3 用 `chat()` 让机器人连续回答问题。机器人说完一个回答后，还可以接着听下一个问题；一轮问答结束时，整个会话通常仍在继续。

连续运行两次启动会话的代码格时，程序应保留正在使用的会话，避免重复启动。运行停止会话的代码格后，还要确认旧任务已经结束，才能再次启动。本课通过分别运行“启动会话、查看状态、停止会话”的代码格，管理同一个持续语音会话。


## 二、持续语音会话的句柄与状态

### 2.1 “听、懂、说”分别处理什么

例如，学生问“刚才提到的语音识别有什么用途？”。一次语音问答涉及以下功能：

| 功能 | 处理的输入 | 产生的输出 | 已有的学习经验 |
|---|---|---|---|
| 采音 | 麦克风接收到的声音 | 音频采样数据 | L1 录音与声音块 |
| 语音转写（ASR） | 音频数据 | 识别出的文字 | L1 `recognize_stream()` |
| 回复生成（LLM） | 提问文字、已有对话与人设要求 | 回复文字 | L3 人设与连续问答 |
| 语音合成（TTS） | 回复文字 | 语音数据 | L2 `synthesize()` |
| 播放 | 语音数据 | 扬声器发出的声音 | L1、L2 播放声音 |

ASR 将声音转为文字，例如转写出“刚才提到的语音识别有什么用途”。其中“刚才提到的”指什么，需要结合前面的对话。LLM 根据这些文字、上下文和人设要求组织回答；TTS 将回答文字合成为声音，最后由扬声器播放。

“听”主要对应采音与转写，“懂”在这里指模型结合问题和上下文生成相关回答，“说”对应合成与播放。转写错误会影响后续回答；转写正确时，模型也可能答非所问。判断交互是否有效，需要结合提问和实际回答。

`chat()` 将这些能力组织为持续语音对话。启动一个会话后，学生可以连续提问。本课通过这个接口使用问答能力，无需逐项调用上表中的接口来拼接会话。表格说明功能关系，各功能在实现时也可能边接收边处理。


### 2.2 `chat()` 返回了什么

L3 中的 `task = speech.chat(...)`，把返回的对象保存在 `task` 中。这个对象叫**任务句柄**：程序凭它查看任务、等待结束或请求取消。可以把句柄理解为这次任务的办理凭据；保留它，才能继续查询和管理这次任务。本课使用变量名 `chat_task` 保存聊天任务的句柄。

官方将这个返回类型写成 `TaskHandle[None]`：

- `TaskHandle` 表示返回的是任务句柄。
- 方括号中的 `None` 表示任务完成时不提供结果数据。机器人仍会说话，`chat()` 也确实返回了句柄。

L2 的 `synthesize()` 会产生音频数据，成功后用 `result()` 取出音频。`chat()` 返回的句柄用于管理整个持续会话，使用 `wait()` 等待任务结束。机器人在会话中说出的回复，不会作为这个句柄的文字结果返回。

句柄的 `trace_id` 是本次任务的追踪标识。后面的程序打印它，便于比较重复运行启动格时是否保留旧任务，以及旧任务结束后再次启动时是否创建了新任务。


### 2.3 从句柄读取任务状态

`chat_task.status` 读取这个会话任务的当前状态。状态名称由 SDK 定义，程序会根据任务的实际执行情况更新它。

| `status` 的取值 | 含义 | `done()` 的返回值 |
|---|---|---|
| `PENDING` | 任务已创建，等待执行 | `False` |
| `RUNNING` | 任务正在执行 | `False` |
| `CANCELLING` | 任务正在取消 | `False` |
| `SUCCEEDED` | 任务成功结束 | `True` |
| `FAILED` | 任务因失败结束 | `True` |
| `CANCELLED` | 任务已取消 | `True` |

最后三种状态称为**终态**。`chat_task.done()` 用来判断任务是否已进入终态，适合用作“是否允许重新开始”的条件。`RUNNING` 描述整个会话正在执行；同一个会话里，机器人听问题、回答问题、等待下次提问，都可能保持这个状态。

| 写法 | 程序执行的操作 |
|---|---|
| `chat_task.status` | 读取当前状态；这个属性只读 |
| `chat_task.done()` | 判断任务是否已结束 |
| `chat_task.error` | 读取错误信息；任务为 `FAILED` 时用它排查 |
| `chat_task.cancel()` | 请求取消任务；返回布尔值，表示请求是否成功提交 |
| `chat_task.wait(timeout=5.0)` | 等待任务结束，最多等五秒；返回结束时或超时时的状态 |

取消请求提交成功后，任务可能还在结束过程中。有限等待返回后仍要检查 `done()`；达到等待上限只会让本次等待返回，不会自动取消任务。


### 2.4 在 Notebook 中使用持续任务

L3 在启动后调用不带超时的 `wait()`，当前代码格会一直等待整个会话结束。本课保存 `chat()` 返回的句柄，让启动代码格先执行完，再在其他代码格中查看和停止同一个任务。

语音任务由 SDK 异步执行：启动调用返回后，任务可以继续运行，Python 也可以执行下一条语句。因此，代码格运行结束时，机器人仍可能继续接收提问。后面的实验将分别观察代码格和会话任务的运行情况。


## 三、观察句柄与任务状态

⚠️ **注意**：本课只使用麦克风和扬声器，不发送运动指令。保持适中音量，各组错开说话；开始前结束 L1～L3 留下的录音、识别和聊天任务，不同时运行多个语音 Notebook。

📌 **前置准备**：完成开机、配网和 IP 确认，电脑与机器人连接同一 WiFi，已能用 BoosterStudio 连接。机器人保持教师确认的稳定姿态，本课不切换运动模式。如果接续行走实验，先按该课流程停止运动、回到 `prepare` 并确认站稳；收起时由人工支撑，再进入 `damping` 并扶坐，严禁从 `walk` 直接切到 `damping`。

### 3.1 连接与初始化

**STEP 01：** 连接机器人，建立保存句柄的变量。

<font color="red">提示：</font>初始化格只在确认没有遗留任务时运行一次。会话启动后，重新执行 `chat_task = None` 会丢失原句柄，后台任务却可能仍在运行。需结束旧任务时，先保留句柄并按下列步骤处理。

> **补充操作：结束遗留会话**
>
> 1. 如果是在另一个Notebook中运行的程序，回到启动旧会话的 Notebook。如果原代码格仍在 `wait()` 中运行，先中断该代码格，再查看它的取消与收尾输出。
> 2. L3 使用变量 `task` 保存句柄。若该变量仍保留旧句柄，可在原 Notebook 的临时代码格运行：
>
>    ```python
>    task.cancel()
>    task.wait(timeout=5.0)
>    print("旧任务状态：", task.status, "；是否结束：", task.done())
>    ```
>
>    若旧会话来自本课，使用它原来所在内核中的 `chat_task`，将以上代码的 `task` 换为 `chat_task`。只有 `done()` 为 `True` 才能继续初始化；仍为 `False` 时保留句柄，由教师继续检查。
> 3. 若原句柄已被覆盖，且没有其他变量保存它，暂停实验，由教师按设备现场维护流程处理遗留会话，确认退出后再继续。重启内核、清空变量或中断代码格，都不能单独证明机器人端任务已经结束。

初始化中的 `chat_task = None` 表示这个变量还没有保存任务句柄。它与返回类型 `TaskHandle[None]` 中表示“没有最终结果数据”的 `None` 用途不同。


In [ ]:
from boosteros.robots.booster import BoosterRobot
from boosteros.brain import Speech
from boosteros.brain.speech import ChatConfig

robot = BoosterRobot()
speech = Speech(robot)
robot.audio_manager.set_system_volume(0.5)

PROMPT = "你是机器人实训课的学习助手。请用简短、清楚的中文回答。"
chat_task = None

print("已连接：", robot.robot_info.model, robot.robot_info.serial_number)


### 3.2 启动并查看同一个会话

**STEP 02：** 发起会话，保存句柄。下一格先检查旧任务；没有旧任务或旧任务已经结束时，才创建新会话。

下一格的 `chat_task = speech.chat(...)` 是保存句柄的赋值语句：等号右侧发起会话并返回句柄，左侧的变量 `chat_task` 保存它。后续查看、等待和取消都通过这个变量操作同一个任务。

代码格执行完后，问“什么是语音识别？”，再问“请举一个刚才这种技术的应用”。

听完回答后暂时不说话，再运行 STEP 03。记录能否连续问答，以及等待提问时的任务状态。


In [ ]:
if chat_task is None or chat_task.done():
    chat_task = speech.chat(
        config=ChatConfig(system_prompt=PROMPT, volume=0.5),
        enable_subtitle_log=True,
    )
    print("已保存新句柄。")
else:
    print("已有未结束的会话，保留原句柄。")

print("任务标识：", chat_task.trace_id)
print("SDK 状态：", chat_task.status)


**STEP 03：** 读取状态，并进行一次有限等待。

`wait(timeout=2.0)` 会阻塞当前代码格，最多等待任务结束两秒。任务提前结束时，等待会提前返回；达到两秒上限时，返回当时的状态。`wait()` 返回后，程序才把这个状态保存到 `observed_status`，再执行后面的 `print()`。

会话持续运行时，等待返回值和当前状态都可能为 `RUNNING`，`done()` 为 `False`：本次等待已经结束，会话任务仍未结束。两秒是等待上限，不是一轮问答的耗时。

`observed_status` 保存本次等待返回的状态，`chat_task.status` 读取打印时的当前状态；两次读取之间若发生状态变化，打印结果可能不同。

记下任务标识，再次运行 STEP 02。若旧任务仍未结束，应出现“已有未结束的会话，保留原句柄”，任务标识与前一次相同。随后再提一个问题，检查原会话能否继续使用。这一步检查重复启动是否被拦截，不要求任务状态发生变化；若任务已自行结束，记录终态和错误信息，再决定是否重开。


In [ ]:
observed_status = chat_task.wait(timeout=2.0)
print("本次等待返回的状态：", observed_status)
print("当前 SDK 状态：", chat_task.status)
print("任务是否结束：", chat_task.done())
print("任务标识：", chat_task.trace_id)
if chat_task.status == "FAILED":
    print("错误信息：", chat_task.error)

### 3.3 用回调观察状态变化

**STEP 04：** 为当前句柄登记一个状态回调。

L1 把函数 `on_text(text)` 交给 SDK，收到识别文字时，由 SDK 调用它并传入文字。本课沿用这种回调方式观察任务状态：把 `on_status(task)` 登记到当前句柄后，SDK 在调用它时传入任务句柄，函数再通过 `task.status` 读取并打印状态。两种回调接收的信息不同，前者接收文字，后者接收句柄。

`add_status_change_callback()` 登记后会立即按当前状态调用一次，之后任务状态发生变化时再调用。这一格对当前句柄只运行一次，重复登记会产生重复输出。

登记完成后继续提问，再暂时保持安静，同时用 STEP 03 查看状态。听问题、回答和等待下次提问，都可能属于同一个 `RUNNING` 任务，因此不一定产生新的状态回调。到 STEP 05 取消任务时，再观察有没有后续输出；短暂状态可能来不及看到，按实际输出记录。

这个回调绑定当前任务，重新创建的任务需要另行登记。


In [ ]:
def on_status(task):
    print("状态回调：", task.trace_id, task.status)

chat_task.add_status_change_callback(on_status)


### 3.4 打断说话与取消会话

**STEP 05：** 在机器人正在回答时，尝试说“停一下”。先观察当前播报是否停止，再运行 STEP 03，记录任务标识、状态和 `done()`；随后再提一个问题，检查能否继续交流。

语音打断若生效，当前播报会停止，而整个会话仍保持 `RUNNING`、`done()` 为 `False`，任务标识也保持不变。此时仍能继续提问，说明打断播报没有结束整个会话。“停一下”是否生效受说话时机、识别和服务行为影响，请分别记录“播报是否停止”“任务是否结束”“能否继续提问”。

然后运行下一格，用 `cancel()` 请求结束整个会话。五秒是这次停止操作愿意等待的上限。若还未进入终态，保留句柄，稍后再运行 STEP 03 查看，不另开会话。


In [ ]:
if chat_task is None:
    print("还没有创建会话。")
elif chat_task.done():
    print("任务已经结束：", chat_task.status)
else:
    accepted = chat_task.cancel()
    print("取消请求是否成功提交：", accepted)
    print("等待返回的状态：", chat_task.wait(timeout=5.0))
    print("任务是否结束：", chat_task.done())
    if not chat_task.done():
        print("尚未确认结束，请保留句柄，稍后查看。")

if chat_task is not None and chat_task.status == "FAILED":
    print("错误信息：", chat_task.error)


## 四、用状态机管理会话

### 4.1 应用状态与转换条件

“任务句柄”提供实际执行状态，“对话应用”还要规定在什么条件下可以执行各项操作。

我们可以用 `app_state` 保存并管理这种状态，**状态机**由有限的状态、触发条件和相应动作组成。

| 应用状态 | 含义 | 操作与转换 |
|---|---|---|
| 待机 | 没有任务，或旧任务已确认结束 | “开始”创建新任务、保存句柄，转为“会话已发起” |
| 会话已发起 | 已保存句柄，尚未确认结束 | 可查看；重复“开始”保留旧任务；取消请求提交成功后转为“停止请求已发出” |
| 停止请求已发出 | 已请求取消，仍需确认结束 | 可查看或继续等待；暂不允许重开；确认 `done()` 为 `True` 后回到“待机” |

`app_state` 由本课程序维护，`chat_task.status` 由 SDK 依据实际执行情况更新。例如，程序调用 `cancel()` 并收到 `True` 后，可以把应用状态改为“停止请求已发出”；随后仍要检查 SDK 任务是否已进入终态。读取状态、调用 `wait()` 或改写 `app_state`，都不会自行结束会话。

即使没有运行停止操作，任务也可能因故障结束。管理程序每次处理操作前都要重新检查句柄；发现 `FAILED` 时显示错误，先请教师核查原因，再决定是否重试。

表中“开始、查看、停止”是传给管理函数的操作文字。将 `command` 设为相应文字并运行代码格，程序才会处理这次操作。

依据表格画出状态图，给箭头标出条件。重复“开始”及取消请求未提交成功时，应保留在哪个状态？


### 4.2 用函数执行状态转换

**STEP 06：** 定义管理函数。函数接收操作文字、旧句柄和应用状态，返回更新后的句柄与状态。

函数先根据句柄核对应用状态，再按“开始、查看、停止”处理操作。“开始”同时要求应用处于待机、旧任务不存在或已经结束。处于“停止请求已发出”时，再次执行“停止”操作会继续等待原任务。

请在代码中找出创建任务、提交取消、阻止重开和回到待机的位置。下面复用了 STEP 02～05 的接口，把它们放进同一个管理函数。


In [ ]:
app_state = "待机"  # 函数运行时会按旧句柄重新核对状态

def manage_chat(command, task, state):
    if task is not None and task.status == "FAILED":
        print("旧任务错误信息：", task.error)
    if task is None or task.done():
        state = "待机"
    elif task.status == "CANCELLING":
        state = "停止请求已发出"
    elif state != "停止请求已发出":
        state = "会话已发起"

    if command == "开始":
        if state == "待机" and (task is None or task.done()):
            task = speech.chat(
                config=ChatConfig(system_prompt=PROMPT, volume=0.5),
                enable_subtitle_log=True,
            )
            state = "会话已发起"
        else:
            print("旧会话尚未结束，保留原句柄。")

    elif command == "停止":
        if state == "待机":
            print("当前没有需要停止的任务。")
        elif state == "停止请求已发出":
            print("已请求停止，继续等待原任务。")
            task.wait(timeout=5.0)
        else:
            accepted = task.cancel()
            print("取消请求是否成功提交：", accepted)
            if accepted:
                state = "停止请求已发出"
            task.wait(timeout=5.0)

    elif command == "查看":
        if task is None:
            print("还没有创建会话。")
    else:
        print("操作文字应为：开始、查看、停止。")

    if task is not None:
        if task.done():
            state = "待机"
        elif task.status == "CANCELLING":
            state = "停止请求已发出"
        print("任务标识：", task.trace_id)
        print("SDK 状态：", task.status, "；是否终态：", task.done())
        if task.status == "FAILED":
            print("错误信息：", task.error)
    print("应用状态：", state)
    return task, state


**STEP 07：** 每次修改下一格的 `command`，运行一次，完成下列操作：

1. “查看”：核对 STEP 05 的旧任务是否已结束。
2. “开始”：确认旧任务已结束后启动新会话，完成问答。
3. 再次“开始”：比较任务标识，检查是否保留原会话；继续提问。
4. “停止”，再“查看”：记录请求结果与任务状态。若仍未结束，继续查看或等待，暂不重开。
5. 确认终态后再次“开始”，检查任务标识并完成问答。

<font color="red">提示：</font>`chat_task, app_state = ...` 分别接收函数返回的句柄和应用状态，两者都要保存。失败任务重新启动前，先记录错误并请教师检查。


In [ ]:
command = "查看"  # 每次改成“开始”“查看”或“停止”，再运行这一格
chat_task, app_state = manage_chat(command, chat_task, app_state)


### 4.3 补写启动条件

**STEP 08：** 将下一格的 `False` 替换为允许启动的条件表达式。需同时满足：

1. `app_state` 为“待机”。
2. 没有旧任务，或旧任务的 `done()` 返回 `True`。

用 `and` 连接这两项要求，用括号把第二项的两种情况放在一起。`done()` 是方法，需要带括号调用；`chat_task is None` 应放在这组 `or` 判断的前面，变量为空时 Python 就会跳过后面的任务检查。

每次先在 STEP 07 将 `command` 设为“查看”并运行，更新应用状态，再运行本格。分别记录会话使用中、停止请求已提交但任务尚未结束、确认结束后的判断结果。取消很快完成时，直接记录实际终态，无需设法让取消停在中间状态。

`can_start` 保存本格运行当时的判断结果，不会随后台任务自动更新；观察条件改变后需重新运行。本格只判断是否允许启动，创建任务仍通过 STEP 07 完成。


In [ ]:
can_start = False  # 请按 §4.3 的要求补写；本格只判断，不创建任务
print("当前是否允许启动：", can_start)


### 4.4 结束练习

**STEP 09：** 发送停止操作，再查看结果。离开前确认 `done()` 为 `True`。

等待后仍未结束时，请保留句柄并联系教师检查。代码格执行完或被中断，都不能单独证明后台会话已经退出。


In [ ]:
chat_task, app_state = manage_chat("停止", chat_task, app_state)
chat_task, app_state = manage_chat("查看", chat_task, app_state)
if chat_task is not None and not chat_task.done():
    print("任务尚未结束，请保留句柄并请教师检查。")


## 五、观察与学习评价

### 5.1 操作记录

第三节先填任务信息，第四节再填写应用状态。

| 操作或观察时机 | 任务标识（末几位） | SDK 状态 / `done()` | 应用状态 | 实际问答或程序提示 |
|---|---|---|---|---|
| 首次开始后 | | | — | |
| 回答完、暂未继续提问时 | | | — | |
| 有限等待返回后 | | | — | |
| 登记状态回调后 | | | — | |
| 尝试语音打断后查看状态，再继续提问 | | | — | |
| 第一次取消并等待后 | | | — | |
| 管理函数开始后 | | | | |
| 重复开始后 | | | | |
| 停止并查看后 | | | | |
| 确认终态后再次开始 | | | | |
| 最后停止并确认终态 | | | | |

语音打断观察：播报是否停止______；任务是否结束______；能否继续提问______。

本次 `cancel()` 返回值：______；最终任务状态：______。
状态回调实际显示了：______。短暂状态没有显示时，按实际情况记录。

### 5.2 检查点

- 能按输入输出解释“听、懂、说”，区分一轮问答与持续会话。
- 能说明句柄及 `None` 的含义，通过 `status`、`done()`、`wait()` 判断任务情况。
- 能给状态图的转换写出条件，保留旧句柄，并在确认终态后重新开始。

### 5.3 学习评价

1. 判断：`TaskHandle[None]` 表示 `chat()` 没有返回对象。（　）
2. 判断：`RUNNING` 表示整个会话正在执行，不能据此确定此刻正在生成回答。（　）
3. 判断：`cancel()` 返回 `True`，或有限 `wait()` 已返回，都足以证明任务已结束。（　）
4. 按“开始 → 重复开始 → 停止 → 查看 → 再次开始”画出应用状态转换；在允许再次开始的箭头上写出判定条件。

### 5.4 结合记录解释

1. 机器人说完一个回答、语音打断、程序取消，对本次会话分别产生了什么影响？用实际问答和任务状态支持你的判断。
2. 为什么“停止请求已发出”还要保留旧句柄？如果任务因故障结束，下一次操作前应先检查哪些信息？


## 六、拓展：模型放在机器人上还是通过网络使用

使用语音对话前，应向教师了解当前服务配置：哪些处理在机器人端完成，哪些通过网络调用服务。使用联网模型服务时，机器人需要把相关请求发送给服务端，再接收返回内容。采音、播放和模型计算可以部署在不同位置。

为什么不把模型都直接装在机器人上？部署时需要权衡以下条件：

| 条件 | 在机器人端运行模型 | 通过网络使用服务 |
|---|---|---|
| 计算与存储 | 受本机处理器、内存和存储限制，需选择适配的模型 | 服务端承担相应模型计算，机器人仍负责采音、播放等本地处理 |
| 连接条件 | 已部署的本地模型可减少对远端连接的依赖 | 服务调用依赖网络连接和服务可用性 |
| 安装与更新 | 需要在设备上部署、维护相应模型 | 模型可在服务端维护，设备配置也需匹配服务 |

两种方式都可能受设备性能、模型能力和实际配置影响。结合教师说明的本课配置，指出一个采用联网服务的理由，以及需要检查的一项连接条件。
